# V0.9D Shared Memory Authority Lab

问题：两个 Agent 共享 Memory 时，真正的权限来自哪里？

本实验回答：Memory URI、IPC payload、历史 Session 观察都不是权限。未来访问必须由当前 `CapabilityEvaluator` 授权。

术语：

- Agent：Capability principal，权限主体。
- Process：runtime identity，运行时身份，不是权限主体。
- Session：durable observation，已经发生过的事实记录。
- Memory URI：Memory 地址，不是权限。
- CapabilityGrant：Kernel 可评估的权限。
- Revocation：当前 evaluator 移除 grant 后，未来访问被拒绝；旧 Session observation 不被删除。


In [ ]:
from pathlib import Path
import sys

repo = Path.cwd()
if not (repo / "agentkernel").exists():
    repo = repo.parent
if str(repo) not in sys.path:
    sys.path.insert(0, str(repo))

from agentkernel import (
    AgentRegistry,
    CapabilityDelegator,
    CapabilityEvaluator,
    CapabilityGrant,
    DelegateCapabilityRequest,
    EventType,
    InMemoryIPCPersistence,
    InMemoryMemoryStore,
    KernelIPC,
    MEMORY_READ_ACTION,
    MEMORY_WRITE_ACTION,
    MemoryAccessDenied,
    MemoryProvenance,
    MemoryService,
    ProcessManager,
    Session,
    memory_namespace_scope,
    project_memories_to_context_pages,
)

AGENT_A = "agent-a"
AGENT_B = "agent-b"
PRIVATE = "private"

def show(title, **items):
    print("\n=== " + title + " ===")
    for key, value in items.items():
        print(f"{key}: {value}")

def grants(agent_id, action, scope):
    return CapabilityEvaluator((CapabilityGrant(agent_id, action, scope),))

def provenance():
    return MemoryProvenance(
        source="host",
        source_class="HOST_VERIFIED",
        source_session_id="session-a",
        source_event_id="event-1",
        source_agent_id=AGENT_A,
    )

def denied(call):
    try:
        call()
    except MemoryAccessDenied as error:
        return True, str(error)
    return False, "allowed"

memory = MemoryService(InMemoryMemoryStore())
session_a = Session("session-a")
session_b = Session("session-b")
registry = AgentRegistry()
agent_a = registry.create_root(agent_id=AGENT_A, session=session_a, record=False)
agent_b = registry.create_root(agent_id=AGENT_B, session=session_b, record=False)
manager = ProcessManager(agent_registry=registry)
process_a = manager.create_process(process_id="process-a", agent=agent_a.control)
process_b = manager.create_process(process_id="process-b", agent=agent_b.control)

show(
    "Setup",
    agent_a=AGENT_A,
    agent_b=AGENT_B,
    process_a=process_a.process_id,
    process_b=process_b.process_id,
    session_a=session_a.session_id,
    session_b=session_b.session_id,
)


## Step 1：Agent A 写入 private Memory

Agent A 有 `memory.write memory://agent-a/private/**`，所以它可以创建自己的 private memory。

In [ ]:
private_scope = memory_namespace_scope(AGENT_A, PRIVATE)
record = memory.remember(
    agent_id=AGENT_A,
    owner_agent_id=AGENT_A,
    namespace=PRIVATE,
    content="Secret roadmap: launch after safety review.",
    provenance=provenance(),
    capability_evaluator=grants(AGENT_A, MEMORY_WRITE_ACTION, private_scope),
)
show("Agent A Memory", memory_id=record.memory_id, memory_uri=record.uri, namespace=record.namespace)


## Step 2：把 Memory URI 通过 IPC 发给 Agent B

注意：IPC payload 只是数据。即使 Agent B 拿到了 URI，也还没有 `memory.read` authority。

In [ ]:
ipc = KernelIPC(
    agent_registry=registry,
    process_manager=manager,
    sessions={AGENT_A: session_a, AGENT_B: session_b},
    persistence=InMemoryIPCPersistence(),
)
ipc.create_channel(channel_id="channel-ab", sender_agent_id=AGENT_A, receiver_agent_id=AGENT_B)
ipc.send(channel_id="channel-ab", sender_process_id=process_a.process_id, payload={"memory_uri": record.uri})
message = ipc.receive(channel_id="channel-ab", receiver_agent_id=AGENT_B)

show(
    "IPC Delivered",
    payload=message.payload,
    sender_agent=message.sender_agent_id,
    receiver_agent=message.receiver_agent_id,
    receiver_session=session_b.session_id,
)

is_denied, reason = denied(lambda: memory.read(record.uri, agent_id=AGENT_B, capability_evaluator=CapabilityEvaluator(())))
show("Agent B Read With URI Only", denied=is_denied, reason=reason)


## Step 3：Agent A 委托 read capability 给 Agent B

现在 authority 发生变化：不是因为 B 有 URI，而是因为 B 的当前 evaluator 里有了 CapabilityGrant。

In [ ]:
parent_grant = CapabilityGrant(AGENT_A, MEMORY_READ_ACTION, record.uri)
decision = CapabilityDelegator().delegate(
    DelegateCapabilityRequest(
        parent_agent_id=AGENT_A,
        child_agent_id=AGENT_B,
        action=MEMORY_READ_ACTION,
        resource_scope=record.uri,
    ),
    parent_grants=(parent_grant,),
)
delegated = CapabilityEvaluator((decision.delegated_grant,))
restored = memory.read(record.uri, agent_id=AGENT_B, capability_evaluator=delegated)

show(
    "Delegated Read",
    delegation_allowed=decision.allowed,
    grant_subject=decision.delegated_grant.subject,
    grant_action=decision.delegated_grant.action,
    grant_scope=decision.delegated_grant.resource_scope,
    restored_content=restored.content,
)


## Step 4：撤销等于当前 evaluator 不再提供 grant

旧的 Session 观察仍然存在，但新的 Memory read/search/context 必须重新授权。

In [ ]:
session_b.append(
    EventType.TOOL_RESULT,
    {"name": "memory.read", "memory_uri": record.uri, "content": restored.content},
)

revoked = CapabilityEvaluator(())
future_denied, reason = denied(lambda: memory.read(record.uri, agent_id=AGENT_B, capability_evaluator=revoked))
fresh_results = memory.search(
    agent_id=AGENT_B,
    owner_agent_id=AGENT_A,
    namespace=None,
    query="Secret roadmap",
    limit=5,
    capability_evaluator=grants(AGENT_B, MEMORY_READ_ACTION, memory_namespace_scope(AGENT_A, "other")),
)
projection = project_memories_to_context_pages(fresh_results, top_k=5)

show(
    "Revocation",
    future_read_denied=future_denied,
    reason=reason,
    old_session_event_count=len(session_b.events),
    old_session_observed_content=session_b.events[-1].data["content"],
    fresh_search_count=len(fresh_results),
    fresh_context_pages=len(projection.pages),
)


## Final

这个实验验证：

- URI / IPC payload 是数据，不是权限。
- Agent 是 capability principal；Process 只是 runtime identity。
- Delegation 使用现有 `CapabilityDelegator`，不引入 MemoryACL。
- Revocation 影响未来访问，不删除历史 Session observation。
- Fresh Context projection 只包含当前授权的 Memory。